# GPT-2 predictive keyboard with hint-masked training

This notebook fine-tunes pretrained GPT-2 on `data/train.src.tok` and evaluates next-word predictions on
`data/devv_eval.csv`. The first-character hint is baked into the **training objective**: at each word
boundary GPT-2 must choose among the root BPE tokens of words whose first character matches the hint, and
the same candidate set is used at inference. No held-out test data is loaded here.

The 124M-parameter `gpt2` checkpoint is used because it fits safely on an 8 GB RTX 3070 with mixed
precision, gradient checkpointing, and gradient accumulation.

For a monitorable, resumable command-line run (tqdm + optional MLflow + atomic checkpoints + contest test
predictions) use `contest1-train-gpt2` instead of the training cells below.


## 1. Environment

Select the Python kernel from `../.venv` before running the notebook. The next cell ensures that the tested Hugging Face dependency version is installed in that environment.


In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import shutil
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "Contest1"):
            if (candidate / "pyproject.toml").is_file() and (
                candidate / "data" / "train.src.tok"
            ).is_file():
                return candidate.resolve()
    raise FileNotFoundError("Could not locate the Contest1 project and its data directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
EXPECTED_VENV = (PROJECT_ROOT.parent / ".venv").resolve()

if Path(sys.prefix).resolve() != EXPECTED_VENV:
    raise RuntimeError(
        f"Select the {EXPECTED_VENV} kernel before continuing; current prefix: {sys.prefix}"
    )

required_transformers_version = "4.57.6"
try:
    installed_transformers_version = version("transformers")
except PackageNotFoundError:
    installed_transformers_version = None

if installed_transformers_version != required_transformers_version:
    uv = shutil.which("uv")
    if uv is not None:
        subprocess.check_call(
            [uv, "pip", "install", "--python", sys.executable, f"transformers=={required_transformers_version}"]
        )
    else:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", f"transformers=={required_transformers_version}"]
        )

print(f"Python: {sys.executable}")
print(f"Project: {PROJECT_ROOT}")


In [ ]:
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
import json
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)

from contest1 import hint_masked_lib
from contest1.hint_masked_lib import (
    HintMaskedPackedDataset,
    build_hint_candidate_tables,
    build_training_lexicon,
    collate_hint_blocks,
    count_pack_tokens,
    dev_answer_coverage,
    evaluate_model,
    hint_masked_loss,
    predict_top_k,
    write_test_predictions,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

@dataclass(frozen=True)
class ExperimentConfig:
    train_path: Path = PROJECT_ROOT / "data" / "train.src.tok"
    dev_path: Path = PROJECT_ROOT / "data" / "devv_eval.csv"
    output_dir: Path = PROJECT_ROOT / "artifacts" / "gpt2-keyboard-hint-masked"
    model_name: str = "gpt2"
    seed: int = 42
    block_size: int = 128
    max_context_tokens: int = 256
    train_batch_size: int = 8
    eval_batch_size: int = 64
    gradient_accumulation_steps: int = 4
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_steps: int = 200
    max_grad_norm: float = 1.0
    epochs: int = 1
    total_epochs: int = 8
    double_descent: bool = False
    max_train_lines: int | None = None
    max_train_steps: int | None = None
    num_workers: int = 0
    top_k: int = 5
    eval_every_steps: int = 2000
    eval_subset_size: int = 2000
    resume: bool = False
    resume_every_steps: int = 1000

cfg = ExperimentConfig()

for required_path in (cfg.train_path, cfg.dev_path):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_fp16 = device.type == "cuda"
if use_fp16:
    gpu = torch.cuda.get_device_properties(0)
    print(f"Device: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
else:
    print("Device: CPU (training will be slow)")

print(cfg)


## 2. Load and inspect the data

The training corpus is streamed one line at a time. The validation CSV is small enough to keep in memory.


In [ ]:
dev_df = pd.read_csv(cfg.dev_path, keep_default_na=False)
required_columns = ["context", "first letter", "answer"]
if list(dev_df.columns) != required_columns:
    raise ValueError(
        f"Expected validation columns {required_columns}, found {list(dev_df.columns)}"
    )
if dev_df[required_columns].eq("").any(axis=None):
    raise ValueError("Validation data contains empty fields")
if not dev_df["first letter"].str.len().eq(1).all():
    raise ValueError("Every first-character hint must contain exactly one character")
if not dev_df.apply(
    lambda row: row["answer"].startswith(row["first letter"]), axis=1
).all():
    raise ValueError("At least one validation answer does not match its hint")

context_lengths = dev_df["context"].str.split().str.len()
display(
    pd.Series(
        {
            "validation rows": len(dev_df),
            "unique answers": dev_df["answer"].nunique(),
            "unique hints": dev_df["first letter"].nunique(),
            "median context words": context_lengths.median(),
            "maximum context words": context_lengths.max(),
        },
        name="value",
    ).to_frame()
)
display(dev_df.head())


## 3. Load pretrained GPT-2 and build the prediction vocabulary

GPT-2 uses byte-pair tokens rather than the corpus's whitespace-delimited words. Every training word is
represented by the BPE tokens of `" " + word`. The predictor restricts the next-token logits at word
boundaries to the **first BPE token** (the word's root) of words that begin with the supplied hint, then
maps the chosen root to its most frequent complete training word for that hint. This same candidate set
is used both during hint-masked training and at inference. If a hint has no training candidates,
prediction backs off to the most frequent training word with that first character (or the hint itself).

The candidate-vocabulary, dataset, loss, and prediction logic lives in `contest1.hint_masked_lib`, shared
with the `contest1-train-gpt2` command.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

model_dtype = torch.float32
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    dtype=model_dtype,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Loaded {cfg.model_name}: {parameter_count / 1e6:.1f}M parameters")


In [ ]:
lexicon = build_training_lexicon(tokenizer, cfg.train_path)
word_counts = lexicon.word_counts
training_lines = lexicon.training_lines
training_tokens = lexicon.training_tokens
word_bpe = lexicon.word_bpe
fallback_by_hint = lexicon.fallback_by_hint
hint_chars = lexicon.hint_chars
hint_to_idx = lexicon.hint_to_idx
candidate_ids_by_hint = lexicon.candidate_ids_by_hint
hint_token_word = lexicon.hint_token_word
predictable_words = lexicon.predictable_words

print(
    f"Training corpus: {training_lines:,} lines, {training_tokens:,} words, "
    f"{len(word_counts):,} word types"
)

dev_candidate_coverage = dev_answer_coverage(
    dev_df["answer"].tolist(),
    dev_df["first letter"].tolist(),
    predictable_words,
    fallback_by_hint,
)
weighted_single_bpe_rate = sum(
    count for word, count in word_counts.items() if len(word_bpe[word]) == 1
) / training_tokens
dev_bpe_lengths = dev_df["answer"].map(
    lambda word: len(tokenizer.encode(" " + word, add_special_tokens=False))
)
format_diagnostics = pd.Series(
    {
        "training words": training_tokens,
        "training words represented by one BPE token": weighted_single_bpe_rate,
        "training words beginning with non-alphanumeric characters": sum(
            count for word, count in word_counts.items() if not word[0].isalnum()
        ),
        "validation answers requiring multiple BPE tokens": float((dev_bpe_lengths > 1).mean()),
        "validation answers with non-alphanumeric hints": float(
            (~dev_df["first letter"].str.isalnum()).mean()
        ),
        "decoder answer coverage": dev_candidate_coverage,
        "word types with cached BPE ids": len(word_bpe),
        "hints with root candidates": len(candidate_ids_by_hint),
    },
    name="value",
)
display(format_diagnostics.to_frame())


### Diagnosis of the first 5,000-step ordinary-LM run

The first run fine-tuned GPT-2 with a plain causal-LM objective and raised dev top-1 from 31.37% to
36.36%, but it still lagged a full-corpus backoff 5-gram (49.06%). Measured causes:

1. **Unequal data exposure** — 5,000 updates consumed only ~14.8% of the estimated 138.5M-BPE-token corpus.
2. **Word/BPE decoding mismatch** — the original decoder accepted only complete single-BPE words, so the
   10.33% of validation answers needing multiple BPE tokens could never be correct.
3. **Task mismatch** — GPT-2 predicted arbitrary next BPE tokens; the contest scores exact next *words*
   given their first character.
4. **Cleaned-text formatting** — training/validation text strips punctuation and writes PTB markers as
   literal `lrb`/`rrb`, far from GPT-2's natural-text pretraining distribution. There is no
   train/validation formatting mismatch between the cleaned files.

The cells below therefore (a) train for one complete corpus epoch by default, (b) use a hint-masked
word-start objective whose candidate set matches inference exactly, and (c) report letter/digit accuracy
separately from the mostly-trivial symbol rows. The shared implementation lives in the `contest1` package;
for a monitorable, resumable command-line run use `contest1-train-gpt2`.


## 4. First-letter-constrained prediction and validation

`predict_top_k` (from `contest1.hint_masked_lib`) applies the exact same candidate set that hint-masked training
uses.


In [ ]:
demo_predictions = predict_top_k(
    model,
    tokenizer,
    dev_df["context"].head(5).tolist(),
    dev_df["first letter"].head(5).tolist(),
    top_k=cfg.top_k,
    batch_size=5,
    candidate_ids_by_hint=candidate_ids_by_hint,
    hint_token_word=hint_token_word,
    fallback_by_hint=fallback_by_hint,
    max_context_tokens=cfg.max_context_tokens,
)
display(
    dev_df.head(5).assign(
        pretrained_prediction=[items[0] for items in demo_predictions]
    )
)


In [ ]:
pretrained_metrics, pretrained_details, pretrained_per_hint = evaluate_model(
    model,
    tokenizer,
    dev_df,
    label="Pretrained GPT-2 (hint-masked decoder)",
    candidate_ids_by_hint=candidate_ids_by_hint,
    hint_token_word=hint_token_word,
    fallback_by_hint=fallback_by_hint,
    predictable_words=predictable_words,
    top_k=cfg.top_k,
    eval_batch_size=cfg.eval_batch_size,
    max_context_tokens=cfg.max_context_tokens,
)
display(pd.DataFrame([pretrained_metrics]))
display(pretrained_per_hint.head(15))


## 5. Hint-masked fine-tuning

At every word boundary the objective is a restricted softmax over the root BPE tokens of words sharing
the target's first letter:

```text
L = -log P(root | context, candidates whose word starts with hint)
```

Continuation BPE tokens, the end-of-text token, and the first word of every line are not trained. Each
word is tokenized separately, the BPE stream is packed into fixed 128-token blocks, and the parallel
hint annotations are collated into per-position label tensors. Use `max_train_steps` for smoke tests and
`epochs`/`double_descent` for full runs.


In [ ]:
used_bpe, used_lines = count_pack_tokens(cfg.train_path, word_bpe, cfg.max_train_lines)
steps_per_epoch = (
    (used_bpe // cfg.block_size) // cfg.train_batch_size
) // cfg.gradient_accumulation_steps
print(
    f"Training stream: {used_lines:,} lines, {used_bpe:,} packed BPE tokens, "
    f"about {steps_per_epoch:,} optimizer updates per epoch"
)

if cfg.num_workers:
    raise ValueError("Hint-masked training requires num_workers=0 (deterministic stream)")
train_dataset = HintMaskedPackedDataset(
    cfg.train_path,
    word_bpe,
    hint_to_idx,
    tokenizer.eos_token_id,
    cfg.block_size,
    cfg.max_train_lines,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train_batch_size,
    num_workers=0,
    pin_memory=use_fp16,
    collate_fn=collate_hint_blocks,
)

sample_ids, sample_roots, sample_hints = next(iter(train_loader))
print(f"Training batch shapes: ids {tuple(sample_ids.shape)}, labels {tuple(sample_roots.shape)}")
active_count = int((sample_roots >= 0).sum().item())
print(f"Active word-start targets in first batch: {active_count}/{sample_roots.numel()}")


In [ ]:
cand_tensors, local_lookup = build_hint_candidate_tables(
    hint_chars,
    candidate_ids_by_hint,
    tokenizer.vocab_size,
    device,
)

batch_ids = sample_ids.to(device)
batch_roots = sample_roots.to(device)
batch_hints = sample_hints.to(device)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16,
    enabled=use_fp16,
):
    check_logits = model(
        input_ids=batch_ids,
        attention_mask=torch.ones_like(batch_ids),
    ).logits
sample_loss, sample_correct, sample_active = hint_masked_loss(
    check_logits,
    batch_roots,
    batch_hints,
    cand_tensors,
    local_lookup,
)
print(
    f"Loss sanity check: loss={float(sample_loss.detach()):.3f}, "
    f"root accuracy={sample_correct / sample_active:.3f} ({sample_active} targets)"
)


In [ ]:
run_epochs = cfg.total_epochs if cfg.double_descent else cfg.epochs
planned_train_steps = cfg.max_train_steps or steps_per_epoch * run_epochs
if planned_train_steps < 1:
    raise ValueError(
        "Training stream is too small to produce one optimizer update; "
        "increase max_train_lines or lower block/batch sizes"
    )

decay_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad and parameter.ndim >= 2
]
no_decay_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad and parameter.ndim < 2
]
optimizer = torch.optim.AdamW(
    [
        {"params": decay_parameters, "weight_decay": cfg.weight_decay},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=cfg.learning_rate,
)

warmup_steps = min(cfg.warmup_steps, max(1, planned_train_steps // 10))
if cfg.double_descent:
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=planned_train_steps,
    )
else:
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=planned_train_steps,
    )
scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)

cfg.output_dir.mkdir(parents=True, exist_ok=True)
resume_path = cfg.output_dir / "resume_state.pt"
best_path = cfg.output_dir / "best_dev.pt"

if cfg.eval_every_steps and cfg.eval_subset_size:
    if cfg.eval_subset_size < len(dev_df):
        dev_subset = dev_df.sample(n=cfg.eval_subset_size, random_state=cfg.seed)
    else:
        dev_subset = dev_df
else:
    dev_subset = None


def save_resume_state() -> None:
    if not cfg.resume:
        return
    temporary = resume_path.with_name(resume_path.name + ".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "global_step": global_step,
            "training_history": training_history,
            "overflow_steps": overflow_steps,
            "data": {
                "model_name": cfg.model_name,
                "block_size": cfg.block_size,
                "train_batch_size": cfg.train_batch_size,
                "gradient_accumulation_steps": cfg.gradient_accumulation_steps,
                "max_train_lines": cfg.max_train_lines,
                "planned_train_steps": planned_train_steps,
            },
        },
        temporary,
    )
    temporary.replace(resume_path)


def run_dev_eval(step: int) -> dict:
    model.eval()
    try:
        metrics, _, _ = evaluate_model(
            model,
            tokenizer,
            dev_subset,
            label="dev",
            candidate_ids_by_hint=candidate_ids_by_hint,
            hint_token_word=hint_token_word,
            fallback_by_hint=fallback_by_hint,
            predictable_words=predictable_words,
            top_k=cfg.top_k,
            eval_batch_size=cfg.eval_batch_size,
            max_context_tokens=cfg.max_context_tokens,
        )
    finally:
        model.train()
    metrics["step"] = step
    return metrics


# Resume support: restore weights/scheduler and rewind the deterministic stream
# by the number of already-consumed micro-batches.
start_step = 0
skip_micro = 0
global_step = 0
overflow_steps = 0
training_history: list[dict] = []
dev_curve: list[dict] = []
best_dev_accuracy: float | None = None

if cfg.resume and resume_path.is_file():
    state = torch.load(resume_path, map_location="cpu", weights_only=False)
    if state["data"].get("planned_train_steps") != planned_train_steps:
        print("WARNING: resume checkpoint was created with a different step budget")
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    global_step = int(state["global_step"])
    overflow_steps = int(state["overflow_steps"])
    training_history = list(state["training_history"])
    skip_micro = global_step * cfg.gradient_accumulation_steps
    start_step = global_step
    print(f"Resumed from global step {global_step}")

if start_step < planned_train_steps:
    model.train()
    model.config.use_cache = False
    optimizer.zero_grad(set_to_none=True)
    progress = tqdm(
        total=planned_train_steps,
        initial=start_step,
        desc="Hint-masked fine-tuning",
        unit=" update",
    )
    accumulated_batches = 0

    for epoch_index in range(run_epochs):
        for input_ids, label_root_ids, label_hint_ids in train_loader:
            if skip_micro > 0:
                skip_micro -= 1
                continue
            input_ids = input_ids.to(device, non_blocking=True)
            label_root_ids = label_root_ids.to(device, non_blocking=True)
            label_hint_ids = label_hint_ids.to(device, non_blocking=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_fp16,
            ):
                logits = model(
                    input_ids=input_ids,
                    attention_mask=torch.ones_like(input_ids),
                ).logits
                loss, batch_correct, batch_active = hint_masked_loss(
                    logits,
                    label_root_ids,
                    label_hint_ids,
                    cand_tensors,
                    local_lookup,
                )
                scaled_loss = loss / cfg.gradient_accumulation_steps
            scaler.scale(scaled_loss).backward()
            accumulated_batches += 1

            if accumulated_batches < cfg.gradient_accumulation_steps:
                continue

            scaler.unscale_(optimizer)
            grad_norm = float(clip_grad_norm_(model.parameters(), cfg.max_grad_norm))
            overflow = not math.isfinite(grad_norm)
            if not overflow:
                scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            accumulated_batches = 0
            if overflow:
                overflow_steps += 1
                continue

            scheduler.step()
            global_step += 1
            training_history.append(
                {
                    "step": global_step,
                    "loss": float(loss.detach()),
                    "root_accuracy": batch_correct / batch_active,
                    "active_targets": batch_active,
                    "learning_rate": scheduler.get_last_lr()[0],
                    "gradient_norm": grad_norm,
                }
            )
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item():.3f}")

            if (
                cfg.resume
                and cfg.resume_every_steps
                and global_step % cfg.resume_every_steps == 0
            ):
                save_resume_state()

            if (
                cfg.eval_every_steps
                and cfg.eval_subset_size
                and dev_subset is not None
                and global_step % cfg.eval_every_steps == 0
            ):
                metrics = run_dev_eval(global_step)
                dev_curve.append(metrics)
                dev_accuracy = float(metrics["top_1_accuracy"])
                if best_dev_accuracy is None or dev_accuracy > best_dev_accuracy:
                    best_dev_accuracy = dev_accuracy
                    torch.save(
                        {"model": model.state_dict(), "metrics": metrics}, best_path
                    )
                with (cfg.output_dir / "dev_curve.json").open("w", encoding="utf-8") as file:
                    json.dump(dev_curve, file, indent=2)
                print(
                    f"  step {global_step}: dev top-1={dev_accuracy:.4f}, "
                    f"masked root acc={batch_correct / batch_active:.4f}"
                )
                save_resume_state()

            if global_step >= planned_train_steps:
                break
        if accumulated_batches:
            optimizer.zero_grad(set_to_none=True)
            accumulated_batches = 0
        if global_step >= planned_train_steps:
            break

    progress.close()
    print(
        f"Finished {global_step:,} optimizer updates "
        f"({overflow_steps:,} overflow-skipped)"
    )
else:
    print("Checkpoint already at the requested number of steps; no training needed.")

# Save final artifacts.
model.save_pretrained(cfg.output_dir, safe_serialization=True)
tokenizer.save_pretrained(cfg.output_dir)
prediction_resources = {
    "candidate_ids_by_hint": candidate_ids_by_hint,
    "hint_token_word": {
        hint: {str(root): word for root, word in mapping.items()}
        for hint, mapping in hint_token_word.items()
    },
    "fallback_by_hint": fallback_by_hint,
    "hint_chars": hint_chars,
}
with (cfg.output_dir / "prediction_resources.json").open("w", encoding="utf-8") as file:
    json.dump(prediction_resources, file, ensure_ascii=False)
with (cfg.output_dir / "training_history.json").open("w", encoding="utf-8") as file:
    json.dump(training_history, file, indent=2)
with (cfg.output_dir / "experiment_config.json").open("w", encoding="utf-8") as file:
    json.dump(
        {key: str(value) if isinstance(value, Path) else value for key, value in asdict(cfg).items()},
        file,
        indent=2,
    )
if dev_curve:
    with (cfg.output_dir / "dev_curve.json").open("w", encoding="utf-8") as file:
        json.dump(dev_curve, file, indent=2)
print(f"Saved fine-tuned model and resources to {cfg.output_dir}")


## 6. Evaluate and analyze the fine-tuned model


In [ ]:
eval_kwargs = dict(
    candidate_ids_by_hint=candidate_ids_by_hint,
    hint_token_word=hint_token_word,
    fallback_by_hint=fallback_by_hint,
    predictable_words=predictable_words,
    top_k=cfg.top_k,
    eval_batch_size=cfg.eval_batch_size,
    max_context_tokens=cfg.max_context_tokens,
)

fine_tuned_metrics, fine_tuned_details, fine_tuned_per_hint = evaluate_model(
    model,
    tokenizer,
    dev_df,
    label="Hint-masked GPT-2 (final step)",
    **eval_kwargs,
)

rows = [pretrained_metrics, fine_tuned_metrics]
best_model_path = cfg.output_dir / "best_dev.pt"
if best_model_path.is_file():
    best_state = torch.load(best_model_path, map_location="cpu", weights_only=False)
    best_model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name, dtype=model_dtype
    ).to(device)
    best_model.config.pad_token_id = tokenizer.pad_token_id
    best_model.config.use_cache = False
    best_model.load_state_dict(best_state["model"])
    best_metrics, _, _ = evaluate_model(
        best_model,
        tokenizer,
        dev_df,
        label="Hint-masked GPT-2 (best dev checkpoint)",
        **eval_kwargs,
    )
    del best_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    rows.append(best_metrics)

comparison = pd.DataFrame(rows)
display(comparison)

cfg.output_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(cfg.output_dir / "validation_summary.csv", index=False)
fine_tuned_details.to_csv(
    cfg.output_dir / "validation_predictions.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_frame = pd.DataFrame(training_history)
if len(history_frame):
    axes[0].plot(history_frame["step"], history_frame["loss"])
    axes[0].set(title="Hint-masked training loss", xlabel="Optimizer update", ylabel="Loss")

accuracy_columns = ["top_1_accuracy", f"top_{cfg.top_k}_accuracy"]
comparison.set_index("model")[accuracy_columns].plot.bar(ax=axes[1], rot=30, legend=False)
axes[1].set(title="Validation accuracy", ylabel="Accuracy", ylim=(0, 1))
plt.tight_layout()
plt.show()

if len(history_frame):
    fig, axis = plt.subplots(figsize=(8, 4))
    axis.plot(history_frame["step"], history_frame["root_accuracy"])
    axis.set(title="Masked root accuracy (training batches)", xlabel="Optimizer update", ylabel="Root accuracy")
    plt.tight_layout()
    plt.show()


In [ ]:
print("Representative correct predictions")
display(
    fine_tuned_details.loc[
        fine_tuned_details["correct"],
        ["context", "first letter", "answer", "prediction"],
    ].sample(n=min(10, int(fine_tuned_details["correct"].sum())), random_state=cfg.seed)
)

print("Representative errors")
errors = fine_tuned_details.loc[
    ~fine_tuned_details["correct"],
    ["context", "first letter", "answer", "prediction", "top_k_predictions"],
]
display(errors.sample(n=min(10, len(errors)), random_state=cfg.seed))


## 7. Reload the saved model and predict one example

This section is self-contained apart from `predict_top_k` (imported from `contest1.hint_masked_lib`): it loads
the tokenizer, model, and prediction resources saved during training, so it also runs after a kernel
restart.


In [ ]:
from pathlib import Path as _Path
import json as _json

import torch as _torch
from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
from transformers import AutoTokenizer as _AutoTokenizer

saved_output_dir = _Path(cfg.output_dir)

saved_tokenizer = _AutoTokenizer.from_pretrained(saved_output_dir)
saved_tokenizer.pad_token = saved_tokenizer.eos_token
saved_tokenizer.padding_side = "left"
saved_tokenizer.truncation_side = "left"

with (saved_output_dir / "prediction_resources.json").open("r", encoding="utf-8") as file:
    saved_resources = _json.load(file)
saved_candidates = {
    hint: [int(token_id) for token_id in token_ids]
    for hint, token_ids in saved_resources["candidate_ids_by_hint"].items()
}
saved_hint_words = {
    hint: {int(root): word for root, word in mapping.items()}
    for hint, mapping in saved_resources["hint_token_word"].items()
}
saved_fallbacks = saved_resources["fallback_by_hint"]

with (saved_output_dir / "experiment_config.json").open("r", encoding="utf-8") as file:
    saved_config = _json.load(file)

saved_model = _AutoModelForCausalLM.from_pretrained(
    saved_output_dir,
    dtype=_torch.float32,
).to(device)
saved_model.config.pad_token_id = saved_tokenizer.pad_token_id
saved_model.config.use_cache = False

example_context = "south korea and the united states on monday"
example_hint = "w"
example_prediction = predict_top_k(
    saved_model,
    saved_tokenizer,
    [example_context],
    [example_hint],
    top_k=int(saved_config["top_k"]),
    batch_size=1,
    candidate_ids_by_hint=saved_candidates,
    hint_token_word=saved_hint_words,
    fallback_by_hint=saved_fallbacks,
    max_context_tokens=int(saved_config["max_context_tokens"]),
)[0]
print(f"Context: {example_context}")
print(f"Hint: {example_hint}")
print(f"Predictions: {example_prediction}")
